In [6]:
!pip install -U langchain langchain-community faiss-cpu sentence-transformers

  Using cached sentence_transformers-5.2.3-py3-none-any.whl.metadata (16 kB)
Using cached sentence_transformers-5.2.3-py3-none-any.whl (494 kB)
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 3.0.1
    Uninstalling sentence-transformers-3.0.1:
      Successfully uninstalled sentence-transformers-3.0.1


In [7]:
!pip install -U langchain-huggingface

In [8]:
!pip install -U "transformers==4.44.2" "accelerate==0.33.0" "sentence-transformers==3.0.1"
!pip install -U langchain-huggingface

  Using cached sentence_transformers-3.0.1-py3-none-any.whl.metadata (10 kB)
Using cached sentence_transformers-3.0.1-py3-none-any.whl (227 kB)
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.2.3
    Uninstalling sentence-transformers-5.2.3:
      Successfully uninstalled sentence-transformers-5.2.3


In [9]:
!pip install sentence-transformers

In [10]:
import os
import json
import numpy as np
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.docstore.document import Document as LC_Document

# ================================
# CONFIG (UPDATED PATHS FOR JUPYTER /work)
# ================================
INSTRUCTOR_MODEL_PATH = "hkunlp/instructor-large"

# Put your JSON here (example: /work/Cardiology_Data/miriad_cardiology.json)
DATA_PATH = "/home/jovyan/work/Cardiology_Data/miriad_cardiology.json"

# Save FAISS index here (example: /work/vectorstore/faiss_index)
VECTORSTORE_DIR = "/home/jovyan/work/vectorstore"
INDEX_PATH = os.path.join(VECTORSTORE_DIR, "faiss_index")

os.makedirs(VECTORSTORE_DIR, exist_ok=True)

# ================================
# BUILD INDEX
# ================================
def main():
    print("🔹 Loading cardiology data...")
    with open(DATA_PATH, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    raw_data = raw_data[:276139]  # limit for testing
    print(f"🔹 Loaded {len(raw_data)} records")

    # Prepare documents
    instruction = "Represent the cardiology answer for retrieval:"
    documents = []

    for row in raw_data:
        answer = row.get("answer")
        if isinstance(answer, str) and answer.strip():
            documents.append(
                LC_Document(
                    page_content=f"{instruction}\n{answer}",
                    metadata=row
                )
            )
    print(f"🔹 Prepared {len(documents)} documents for embedding")

    # Load embedding model
    print("🔹 Loading Instructor embedding model...")
    embeddings_model = HuggingFaceEmbeddings(model_name=INSTRUCTOR_MODEL_PATH)

    # OPTIONAL: check embedding of first document
    first_embedding = embeddings_model.embed_documents([documents[0].page_content])
    print(f"🔹 Sample embedding shape (first document): {np.array(first_embedding).shape}")

    # OPTIONAL: check embedding shapes for all docs
    all_embeddings = embeddings_model.embed_documents([doc.page_content for doc in documents])
    print(f"🔹 All embeddings shape: {np.array(all_embeddings).shape}")  # (num_docs, embedding_dim)

    # Build FAISS index
    print("🔹 Building FAISS index...")
    vectorstore = FAISS.from_documents(documents, embeddings_model)

    # Save index
    vectorstore.save_local(INDEX_PATH)
    print(f"\n✅ FAISS index successfully saved to: {INDEX_PATH}")

# ================================
# ENTRY POINT
# ================================
if __name__ == "__main__":
    main()


🔹 Loading cardiology data...
🔹 Loaded 276139 records
🔹 Prepared 276139 documents for embedding
🔹 Loading Instructor embedding model...


pytorch_model.bin:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

2_Dense/pytorch_model.bin:   0%|          | 0.00/3.15M [00:00<?, ?B/s]

/opt/conda/lib/python3.11/site-packages/sentence_transformers/models/Dense.py:89: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(input_path, "pytorch_

🔹 Sample embedding shape (first document): (1, 768)
🔹 All embeddings shape: (276139, 768)
🔹 Building FAISS index...

✅ FAISS index successfully saved to: /home/jovyan/work/vectorstore/faiss_index
